In [1]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. HÀM LÀM SẠCH SỐ ĐIỆN THOẠI
# ==========================================
def clean_phone(phone):
    if pd.isna(phone) or str(phone).strip() == '':
        return np.nan
    p = str(phone).split('.')[0]
    p = re.sub(r'\D', '', p)
    if not p:
        return np.nan
    if p.startswith('84'):
        p = '0' + p[2:]
    elif not p.startswith('0'):
        p = '0' + p
    return p

# ==========================================
# 2. ĐỌC DỮ LIỆU THÔ
# ==========================================
print("Đang đọc dữ liệu...")
df_lead = pd.read_csv('/Users/tranvomanhtuan/Downloads/Sales_Marketing dataset - SALE T1-2-3_2026.csv')
df_hen = pd.read_csv('/Users/tranvomanhtuan/Downloads/Sales_Marketing dataset - ĐĂT HẸN .csv')
df_hoadon = pd.read_csv('/Users/tranvomanhtuan/Downloads/Sales_Marketing dataset - DanhSachChiTietHoaDon_KV2502202.csv')

# Lọc các cột cần thiết cho Lead ngay từ đầu
cols_lead = ['SỐ ĐT', 'LOẠI TIN NHẮN', 'NHÓM SP', 'CHATPAGE', 'NGUỒN', 'QUAN TÂM', 'TÌNH TRẠNG']
cols_lead = [c for c in cols_lead if c in df_lead.columns]
df_lead = df_lead[cols_lead]

# ==========================================
# 3. CHUẨN HÓA CỘT KEY (SỐ ĐIỆN THOẠI) & TÁCH DATA
# ==========================================
df_lead['Phone_Clean'] = df_lead['SỐ ĐT'].apply(clean_phone)
df_hen['Phone_Clean'] = df_hen['SỐ ĐT'].apply(clean_phone)
df_hoadon['Phone_Clean'] = df_hoadon['Điện thoại'].apply(clean_phone)

# TÁCH RIÊNG NHỮNG LEAD KHÔNG CÓ SĐT ĐỂ THÊM VÀO SAU
df_lead_no_phone = df_lead[df_lead['Phone_Clean'].isna()].copy()
df_lead_no_phone['In_Lead'] = True
df_lead_no_phone['Phân nhóm MECE'] = 'Nhóm 0: Lead chưa có SĐT'
df_lead_no_phone['Số lượng Lead tính'] = 1 # Đếm là 1 Lead

# CHỈ GIỮ LẠI CÁC DÒNG CÓ SĐT ĐỂ MERGE
df_lead_has_phone = df_lead.dropna(subset=['Phone_Clean'])
df_hen = df_hen.dropna(subset=['Phone_Clean'])
df_hoadon = df_hoadon.dropna(subset=['Phone_Clean'])

# Lọc cột cho Hẹn và Hóa đơn
cols_hen = ['Phone_Clean', 'NGÀY HẸN']
cols_hen = [c for c in cols_hen if c in df_hen.columns]

cols_hoadon = ['Phone_Clean', 'Mã hóa đơn', 'Mã khách hàng', 'Điện thoại', 'Khách cần trả', 'Tên hàng', 'Thời gian']
cols_hoadon = [c for c in cols_hoadon if c in df_hoadon.columns]

df_hen = df_hen[cols_hen]
df_hoadon = df_hoadon[cols_hoadon]

# ==========================================
# 4. XỬ LÝ TRÙNG LẶP TRƯỚC KHI MERGE
# ==========================================
df_lead_unique = df_lead_has_phone.drop_duplicates(subset=['Phone_Clean'], keep='first').copy()
df_lead_unique['In_Lead'] = True  

df_hen_unique = df_hen.drop_duplicates(subset=['Phone_Clean'], keep='first').copy()
df_hen_unique['In_Hen'] = True    

# ==========================================
# 5. GỘP DỮ LIỆU CÓ SĐT (OUTER JOIN)
# ==========================================
df_merge_1 = pd.merge(df_lead_unique, df_hoadon, on='Phone_Clean', how='outer')
df_final_has_phone = pd.merge(df_merge_1, df_hen_unique, on='Phone_Clean', how='outer')

# ==========================================
# 6. PHÂN NHÓM MECE CHO DATA CÓ SĐT
# ==========================================
def phan_nhom_mece(row):
    L = row['In_Lead'] == True
    H = row['In_Hen'] == True
    HD = pd.notna(row['Mã hóa đơn'])
    
    if not L and not H and HD:
        return 'Nhóm 1: Khách tự nhiên vãng lai (Chỉ có HĐ)'
    elif L and not H and not HD:
        return 'Nhóm 2: Lead có SĐT nhưng chưa hẹn & chưa chốt'
    elif L and not H and HD:
        return 'Nhóm 3: Lead chốt thẳng không cần hẹn'
    elif L and H and not HD:
        return 'Nhóm 4: Lead đặt hẹn nhưng rớt (bom lịch/ko mua)'
    elif L and H and HD:
        return 'Nhóm 5: Lead hoàn hảo (Đủ 3 bước)'
    else:
        return 'Khác'

df_final_has_phone['Phân nhóm MECE'] = df_final_has_phone.apply(phan_nhom_mece, axis=1)

# Xử lý đếm Lead trùng lặp cho khách có SĐT
if 'Thời gian' in df_final_has_phone.columns:
    df_final_has_phone = df_final_has_phone.sort_values(by=['Phone_Clean', 'Thời gian'], na_position='last')
df_final_has_phone['Số lượng Lead tính'] = (~df_final_has_phone.duplicated(subset=['Phone_Clean'])).astype(int)

# Đồng bộ hiển thị SĐT
df_final_has_phone['SĐT Cuối'] = df_final_has_phone['Phone_Clean'].apply(lambda x: f"'{x}" if pd.notna(x) else np.nan)

# ==========================================
# 7. GỘP LẠI (CONCAT) VỚI NHỮNG LEAD KHÔNG CÓ SĐT
# ==========================================
# Cột SĐT Cuối của Nhóm 0 sẽ để trống
df_final = pd.concat([df_final_has_phone, df_lead_no_phone], ignore_index=True)

# ==========================================
# 8. DỌN DẸP VÀ XUẤT FILE
# ==========================================
columns_to_drop = ['In_Lead', 'In_Hen', 'Điện thoại', 'Phone_Clean', 'SỐ ĐT']
df_final = df_final.drop(columns=[c for c in columns_to_drop if c in df_final.columns])

# Đưa cột quan trọng lên đầu
cols = df_final.columns.tolist()
cols_first = ['SĐT Cuối', 'Phân nhóm MECE', 'Số lượng Lead tính']
cols = cols_first + [c for c in cols if c not in cols_first]
df_final = df_final[cols]

# Xuất ra Excel
file_output = '/Users/tranvomanhtuan/Downloads/BaoCao_Gop_PhanTich_MECE.xlsx'
df_final.to_excel(file_output, index=False)
print(f"Xử lý thành công! File đã lưu tại: {file_output}")

Đang đọc dữ liệu...
Xử lý thành công! File đã lưu tại: /Users/tranvomanhtuan/Downloads/BaoCao_Gop_PhanTich_MECE.xlsx


In [2]:
# ==========================================
# 9. BÁO CÁO KIỂM TOÁN DỮ LIỆU (DATA AUDIT REPORT)
# ==========================================
print("\n" + "="*50)
print("BÁO CÁO NGHIỆM THU MERGE DỮ LIỆU")
print("="*50)

# 1. ĐỐI SOÁT VĨ MÔ (Đảm bảo không bị rớt dữ liệu)
print("\n[1] KIỂM TRA TỔNG QUAN (MACRO)")

# Kiểm tra Hóa đơn (Quan trọng nhất: Không được rớt doanh thu)
hd_ban_dau = df_hoadon['Mã hóa đơn'].nunique()
hd_sau_merge = df_final_has_phone['Mã hóa đơn'].dropna().nunique()
print(f"- Tổng số Mã hóa đơn ở file gốc: {hd_ban_dau}")
print(f"- Tổng số Mã hóa đơn sau merge:  {hd_sau_merge}")
if hd_ban_dau == hd_sau_merge:
    print("  => CHUẨN XÁC: Không rớt bất kỳ hóa đơn nào!")
else:
    print(f"  => CẢNH BÁO: Lệch {abs(hd_ban_dau - hd_sau_merge)} hóa đơn!")

# Kiểm tra Lead có SĐT
lead_ban_dau = df_lead_unique['Phone_Clean'].nunique()
lead_sau_merge = df_final_has_phone[df_final_has_phone['Phân nhóm MECE'].str.contains('Lead')]['Phone_Clean'].nunique()
print(f"\n- Tổng số Lead (có SĐT) file gốc: {lead_ban_dau}")
print(f"- Tổng số Lead (có SĐT) sau merge: {lead_sau_merge}")
if lead_ban_dau == lead_sau_merge:
    print("  => CHUẨN XÁC: Không rớt SĐT Lead nào!")

# 2. KIỂM TRA LOGIC MECE
print("\n[2] PHÂN BỔ 6 NHÓM MECE")
print(df_final['Phân nhóm MECE'].value_counts().to_string())

# 3. TRUY VẾT CÁ NHÂN (MICRO)
print("\n[3] CÔNG CỤ TRUY VẾT SĐT NGẪU NHIÊN (MICRO)")

def truy_vet_sdt(sdt_can_tim):
    sdt_clean = clean_phone(sdt_can_tim)
    print(f"\nĐang truy vết SĐT: {sdt_can_tim} (Clean: '{sdt_clean}')")
    
    # Check sự tồn tại trong 3 file gốc
    in_lead = sdt_clean in df_lead_has_phone['Phone_Clean'].values
    in_hen = sdt_clean in df_hen['Phone_Clean'].values
    in_hd = sdt_clean in df_hoadon['Phone_Clean'].values
    
    print(f"- Tồn tại trong file Lead (Sale T1-2-3)? : {'CÓ' if in_lead else 'Không'}")
    print(f"- Tồn tại trong file Đặt Hẹn?            : {'CÓ' if in_hen else 'Không'}")
    print(f"- Tồn tại trong file Hóa Đơn?            : {'CÓ' if in_hd else 'Không'}")
    
    # Check kết quả sau merge
    ket_qua = df_final[df_final['Phone_Clean'] == sdt_clean]
    if not ket_qua.empty:
        nhom_mece = ket_qua['Phân nhóm MECE'].iloc[0]
        so_hd = ket_qua['Mã hóa đơn'].dropna().nunique()
        print(f"=> KẾT LUẬN MERGE: Khách được xếp đúng vào: [{nhom_mece}] với {so_hd} hóa đơn.")
    else:
        print("=> KẾT LUẬN: Không tìm thấy trong file kết quả!")

# Máy tính tự động bốc ngẫu nhiên 3 SĐT ở 3 nhóm khác nhau để test chéo
try:
    sdt_test_nhom1 = df_final[df_final['Phân nhóm MECE'].str.contains('Nhóm 1')]['Phone_Clean'].dropna().sample(1).iloc[0]
    sdt_test_nhom4 = df_final[df_final['Phân nhóm MECE'].str.contains('Nhóm 4')]['Phone_Clean'].dropna().sample(1).iloc[0]
    sdt_test_nhom5 = df_final[df_final['Phân nhóm MECE'].str.contains('Nhóm 5')]['Phone_Clean'].dropna().sample(1).iloc[0]

    truy_vet_sdt(sdt_test_nhom1) # Khách vãng lai
    truy_vet_sdt(sdt_test_nhom4) # Khách đặt hẹn rớt
    truy_vet_sdt(sdt_test_nhom5) # Khách hoàn hảo
except Exception as e:
    print("\nKhông đủ data để test ngẫu nhiên, hoặc có nhóm không có SĐT nào.")


BÁO CÁO NGHIỆM THU MERGE DỮ LIỆU

[1] KIỂM TRA TỔNG QUAN (MACRO)
- Tổng số Mã hóa đơn ở file gốc: 1003
- Tổng số Mã hóa đơn sau merge:  1003
  => CHUẨN XÁC: Không rớt bất kỳ hóa đơn nào!

- Tổng số Lead (có SĐT) file gốc: 1304
- Tổng số Lead (có SĐT) sau merge: 1304
  => CHUẨN XÁC: Không rớt SĐT Lead nào!

[2] PHÂN BỔ 6 NHÓM MECE
Phân nhóm MECE
Nhóm 0: Lead chưa có SĐT                            1398
Nhóm 2: Lead có SĐT nhưng chưa hẹn & chưa chốt      1018
Nhóm 1: Khách tự nhiên vãng lai (Chỉ có HĐ)          744
Nhóm 5: Lead hoàn hảo (Đủ 3 bước)                    161
Nhóm 3: Lead chốt thẳng không cần hẹn                 98
Nhóm 4: Lead đặt hẹn nhưng rớt (bom lịch/ko mua)      95
Khác                                                   4

[3] CÔNG CỤ TRUY VẾT SĐT NGẪU NHIÊN (MICRO)

Không đủ data để test ngẫu nhiên, hoặc có nhóm không có SĐT nào.


In [3]:
import pandas as pd
import numpy as np

# Đọc file kết quả (Hãy đảm bảo tên file đúng với file trên máy bạn)
df = pd.read_excel('/Users/tranvomanhtuan/Downloads/BaoCao_Gop_PhanTich_MECE.xlsx')

# ==========================================
# 1. CHUẨN HÓA DOANH THU (KHÁCH CẦN TRẢ)
# ==========================================
def clean_money(val):
    if pd.isna(val): return 0
    return int(str(val).replace('.', '').strip())
df['Doanh Thu (VNĐ)'] = df['Khách cần trả'].apply(clean_money)

# ==========================================
# 2. BÙ ĐẮP DỮ LIỆU NHÓM 1
# ==========================================
# Nhóm 1 (Khách vãng lai) sẽ bị thiếu thông tin ở một số cột
idx_nhom1 = df['Phân nhóm MECE'].str.contains('Nhóm 1', na=False)

# Cột loại tin nhắn điền 'Khác'
df.loc[idx_nhom1 & df['LOẠI TIN NHẮN'].isna(), 'LOẠI TIN NHẮN'] = 'Khác'
# Cột nguồn điền 'Khác'
df.loc[idx_nhom1 & df['NGUỒN'].isna(), 'NGUỒN'] = 'Khác'

# Cột NHÓM SP: Suy luận từ Tên Hàng nếu để trống
def fill_nhom_sp(row):
    if pd.notna(row['NHÓM SP']):
        return str(row['NHÓM SP']).strip().upper()
    ten_hang = str(row['Tên hàng']).upper()
    if 'KHÓA HỌC' in ten_hang or 'ĐÀO TẠO' in ten_hang or 'PLASMA ONLINE' in ten_hang:
        return 'ĐÀO TẠO'
    return 'DỊCH VỤ'
df['NHÓM SP_Clean'] = df.apply(fill_nhom_sp, axis=1)

# ==========================================
# 3. GOM NHÓM NGUỒN KHÁCH HÀNG
# ==========================================
def phan_loai_nguon(row):
    chatpage = str(row['CHATPAGE']).upper() if pd.notna(row['CHATPAGE']) else ""
    nguon = str(row['NGUỒN']).upper() if pd.notna(row['NGUỒN']) else ""
    mece = str(row['Phân nhóm MECE']).upper()
    
    combined = chatpage + " | " + nguon
    
    # Facebook
    if any(x in combined for x in ['FB CÔ HƯỜNG', 'FANPAGE PXV', 'FANPAGE HỌC VIỆN', 'FANPAGE', 'FACEBOOK', 'FB']):
        return 'Facebook'
    # Tiktok
    if any(x in combined for x in ['TIKTOK PXV', 'TIKTOK HỌC VIỆN', 'TIKTOK']):
        return 'Tiktok'
    # Hotline/Zalo
    if any(x in combined for x in ['HOTLINE', 'ZALO']):
        return 'Hotline/Zalo'
    # Khách cũ
    if any(x in combined for x in ['KHÁCH CŨ', 'DATA CŨ 2023', 'FILE CŨ']):
        return 'Khách cũ'
    # Vãng lai (nếu tự ghi là vãng lai, hoặc thuộc Nhóm 1 của MECE)
    if any(x in combined for x in ['VÃNG LAI']) or ('NHÓM 1' in mece):
        return 'Vãng lai'
    # Khác
    if any(x in combined for x in ['PAGE LX', 'INSTAGRAM', 'INSTGRAM']):
        return 'Khác'
        
    return 'Khác'
df['Kênh Tiếp Cận'] = df.apply(phan_loai_nguon, axis=1)

# ==========================================
# 4. CHUẨN HÓA SẢN PHẨM & LỌC DỊCH VỤ PHỄU
# ==========================================
def lay_sp_cot_loi(row):
    sp = str(row['Tên hàng']) if pd.notna(row['Tên hàng']) else str(row['QUAN TÂM'])
    sp = sp.upper()
    
    if 'XOÁ' in sp or 'XÓA' in sp: return 'Xóa Laser'
    if 'MÀY' in sp or 'TẠO SỢI' in sp or 'ĐIÊU KHẮC' in sp: return 'Làm Mày'
    if 'MÔI' in sp: return 'Làm Môi'
    if 'CO2' in sp or 'BÓC TÁCH' in sp: return 'CO2 / Cắt đáy sẹo'
    if 'PLASMA' in sp and 'KHÓA' in sp: return 'Khóa học'
    if 'MÍ' in sp or 'EYELINER' in sp: return 'Làm Mí'
    if pd.isna(row['Tên hàng']) and pd.isna(row['QUAN TÂM']): return 'Chưa rõ'
    return 'Dịch vụ/Sản phẩm khác'

# Cột Phân loại sản phẩm (giữ nguyên cột Tên hàng gốc)
df['Phân loại sản phẩm'] = df.apply(lay_sp_cot_loi, axis=1)

# Lọc Dịch vụ phễu (Trả về True/False)
funnel_services = [
    '(DV) GÓI TIẾT KIỆM - XÓA CHÂN MÀY TRỌN GÓI',
    '(DV) Xóa lần 01 - Chân Mày',
    '(DV) Công nghệ CO2 Fractional - Tách thâm',
    '(DV) Công nghệ CO2 Fractional - Gói Double bóc tách',
    '(DV) Công nghệ CO2 Fractional - Gói Double bóc tách (trọn gói)'
]
df['[Dịch vụ phễu]'] = df['Tên hàng'].isin(funnel_services)

# ==========================================
# 5. TẠO CHỈ SỐ PHỄU (FUNNEL METRICS) 1/0
# ==========================================
def tinh_pheu(row):
    if row['Số lượng Lead tính'] == 0:
        return pd.Series([0, 0, 0, 0]) # Bỏ qua các dòng hóa đơn lặp lại của cùng 1 người
    
    b1_mess = 1
    b2_sdt = 0 if 'Nhóm 0' in str(row['Phân nhóm MECE']) else 1
    b3_hen = 1 if ('Nhóm 4' in str(row['Phân nhóm MECE']) or 'Nhóm 5' in str(row['Phân nhóm MECE']) or pd.notna(row['NGÀY HẸN'])) else 0
    b4_don = 1 if ('Nhóm 3' in str(row['Phân nhóm MECE']) or 'Nhóm 5' in str(row['Phân nhóm MECE']) or 'Nhóm 1' in str(row['Phân nhóm MECE']) or pd.notna(row['Mã hóa đơn'])) else 0
    
    return pd.Series([b1_mess, b2_sdt, b3_hen, b4_don])

df[['[F] 1_Có Inbox', '[F] 2_Có SĐT', '[F] 3_Có Đặt Lịch', '[F] 4_Có Ra Đơn']] = df.apply(tinh_pheu, axis=1)

# ==========================================
# LƯU FILE KẾT QUẢ
# ==========================================
file_output = '/Users/tranvomanhtuan/Downloads/Data_Dashboard_Ready_v2.xlsx' 
# Nhớ đổi đường dẫn tuyệt đối như hồi nãy nếu chạy trên Macbook nhé:
# file_output = '/Users/Tên_Của_Bạn/Downloads/Data_Dashboard_Ready_v2.xlsx'

df.to_excel(file_output, index=False)
print(f"Xử lý thành công! Đã lưu file: {file_output}")

Xử lý thành công! Đã lưu file: /Users/tranvomanhtuan/Downloads/Data_Dashboard_Ready_v2.xlsx


In [1]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. HÀM LÀM SẠCH SỐ ĐIỆN THOẠI
# ==========================================
def clean_phone(phone):
    if pd.isna(phone) or str(phone).strip() == '':
        return np.nan
    p = str(phone).split('.')[0]
    p = re.sub(r'\D', '', p)
    if not p:
        return np.nan
    if p.startswith('84'):
        p = '0' + p[2:]
    elif not p.startswith('0'):
        p = '0' + p
    return p

# ==========================================
# 2. ĐỌC DỮ LIỆU
# ==========================================
print("Đang đọc dữ liệu...")
df_dashboard = pd.read_excel('/Users/tranvomanhtuan/Downloads/Data_Dashboard_Ready_v2.xlsx') 
df_lead = pd.read_csv('/Users/tranvomanhtuan/Downloads/Sales_Marketing dataset - SALE T1-2-3_2026.csv')

# ==========================================
# 3. TẠO KEY KHỚP SĐT CHO 2 BẢNG
# ==========================================
df_dashboard['Phone_Key'] = df_dashboard['SĐT Cuối'].apply(clean_phone)
df_lead['Phone_Key'] = df_lead['SỐ ĐT'].apply(clean_phone)

# ==========================================
# 4. TRÍCH XUẤT VÀ ÉP BUỘC ĐỊNH DẠNG NGÀY LEAD (CHỈ 2026)
# ==========================================
df_lead_date = df_lead.dropna(subset=['Phone_Key', 'NGÀY']).copy()

def parse_lead_date_strict(date_str):
    try:
        d_str = str(date_str).strip()
        # Bước 1: Parse lấy ngày và tháng (ưu tiên dd/mm/yyyy hoặc dd/mm)
        d = pd.to_datetime(d_str, format='%d/%m/%Y', errors='coerce')
        if pd.isnull(d):
            d = pd.to_datetime(d_str, format='%d/%m', errors='coerce')
        if pd.isnull(d):
            # Fallback cho các định dạng ngày khác
            d = pd.to_datetime(d_str, dayfirst=True, errors='coerce')
            
        # Bước 2: ÉP BUỘC năm là 2026 (Diệt trừ lỗi gõ nhầm 2025)
        if pd.notnull(d):
            return d.replace(year=2026)
        return pd.NaT
    except:
        return pd.NaT

df_lead_date['Ngay_Lead_Original'] = df_lead_date['NGÀY'].apply(parse_lead_date_strict)

# Bỏ dòng lỗi ngày, sắp xếp và chỉ lấy ngày Lead TỚI SỚM NHẤT của 1 SĐT
df_lead_date = df_lead_date.dropna(subset=['Ngay_Lead_Original'])
df_lead_date = df_lead_date.sort_values('Ngay_Lead_Original').drop_duplicates('Phone_Key', keep='first')
df_lead_date = df_lead_date[['Phone_Key', 'Ngay_Lead_Original']]

# ==========================================
# 5. MERGE VÀO DASHBOARD VÀ TÍNH THỜI GIAN
# ==========================================
df_final = pd.merge(df_dashboard, df_lead_date, on='Phone_Key', how='left')

# Chuyển cột Thời gian (Ngày ra đơn) về datetime
# Lưu ý: Nếu hóa đơn cũng bị gõ nhầm 2025, ta cũng ép nó về 2026 để trừ không bị âm/sai
def fix_hoadon_year(hd_time):
    try:
        d = pd.to_datetime(hd_time, format='%d/%m/%Y %H:%M:%S', errors='coerce')
        if pd.notnull(d) and d.year == 2025:
            return d.replace(year=2026)
        return d
    except:
        return pd.NaT

df_final['Ngay_Hoa_Don'] = df_final['Thời gian'].apply(fix_hoadon_year)

# Tính Thời gian ra đơn (Ngày)
df_final['Thời gian ra đơn (Ngày)'] = (df_final['Ngay_Hoa_Don'] - df_final['Ngay_Lead_Original']).dt.total_seconds() / (24 * 3600)

# Làm tròn 1 chữ số thập phân
df_final['Thời gian ra đơn (Ngày)'] = df_final['Thời gian ra đơn (Ngày)'].round(1)

# Fix số âm thành 0
df_final.loc[df_final['Thời gian ra đơn (Ngày)'] < 0, 'Thời gian ra đơn (Ngày)'] = 0

# ==========================================
# 6. DỌN DẸP VÀ XUẤT FILE
# ==========================================
match_count = df_final['Thời gian ra đơn (Ngày)'].notna().sum()

columns_to_drop = ['Phone_Key', 'Ngay_Hoa_Don']
df_final = df_final.drop(columns=[c for c in columns_to_drop if c in df_final.columns])

file_out = '/Users/tranvomanhtuan/Downloads/Data_Dashboard_Final_With_Time.xlsx'
df_final.to_excel(file_out, index=False)

print(f"\nKiểm tra: Đã match thành công {match_count} đơn hàng có lịch sử Lead!")
print(f"File đã sẵn sàng tại: {file_out}")

Đang đọc dữ liệu...

Kiểm tra: Đã match thành công 258 đơn hàng có lịch sử Lead!
File đã sẵn sàng tại: /Users/tranvomanhtuan/Downloads/Data_Dashboard_Final_With_Time.xlsx


In [1]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. HÀM LÀM SẠCH VÀ CHUẨN HÓA DỮ LIỆU
# ==========================================
def clean_phone(phone):
    """Làm sạch SĐT để làm Key merge chính xác tuyệt đối"""
    if pd.isna(phone) or str(phone).strip() == '': return np.nan
    p = str(phone).split('.')[0]
    p = re.sub(r'\D', '', p)
    if not p: return np.nan
    if p.startswith('84'): p = '0' + p[2:]
    elif not p.startswith('0'): p = '0' + p
    return p

def clean_money(val):
    """Làm sạch tiền tệ từ string sang int"""
    if pd.isna(val): return 0
    return int(str(val).replace('.', '').strip())

def parse_lead_date_strict(date_str):
    """Bắt buộc năm của Lead phải là 2026"""
    try:
        d_str = str(date_str).strip()
        d = pd.to_datetime(d_str, format='%d/%m/%Y', errors='coerce')
        if pd.isnull(d): d = pd.to_datetime(d_str, format='%d/%m', errors='coerce')
        if pd.isnull(d): d = pd.to_datetime(d_str, dayfirst=True, errors='coerce')
        if pd.notnull(d): return d.replace(year=2026)
        return pd.NaT
    except: return pd.NaT

def parse_hoadon_date_strict(hd_time):
    """Tôn trọng định dạng thời gian đã chuẩn hóa của bạn và chỉ chặn năm 2025"""
    try:
        # Lược bỏ tham số 'format', để Pandas tự động nhận diện định dạng bạn đã làm
        d = pd.to_datetime(hd_time, errors='coerce')
        
        # Vẫn giữ lại chốt chặn an toàn: Nếu năm là 2025 thì ép về 2026
        if pd.notnull(d) and d.year == 2025: 
            return d.replace(year=2026)
        return d
    except: 
        return pd.NaT

# ==========================================
# 2. ĐỌC DỮ LIỆU THÔ
# ==========================================
print("Đang đọc và làm sạch dữ liệu thô...")
df_lead = pd.read_csv('/Users/tranvomanhtuan/Downloads/Sales_Marketing dataset.csv')
df_hen = pd.read_csv('/Users/tranvomanhtuan/Downloads/ĐĂT HẸN .csv')
df_hoadon = pd.read_csv('/Users/tranvomanhtuan/Downloads/kiotviet.csv')

# Lọc cột Lead
cols_lead = ['NGÀY', 'SDT', 'LOẠI TIN NHẮN', 'NHÓM SP', 'CHATPAGE', 'NGUỒN', 'QUAN TÂM', 'TÌNH TRẠNG']
df_lead = df_lead[[c for c in cols_lead if c in df_lead.columns]]

# ==========================================
# 3. CHUẨN HÓA KEY (SỐ ĐIỆN THOẠI) & THỜI GIAN TRƯỚC KHI MERGE
# ==========================================
df_lead['Phone_Clean'] = df_lead['SDT'].apply(clean_phone)
df_hen['Phone_Clean'] = df_hen['SỐ ĐT'].apply(clean_phone)
df_hoadon['Phone_Clean'] = df_hoadon['Điện thoại'].apply(clean_phone)

# Chuẩn hóa thời gian từ file gốc
df_lead['Ngay_Lead_Chuẩn'] = df_lead['NGÀY'].apply(parse_lead_date_strict)
df_hoadon['Thời_gian_Chuẩn'] = df_hoadon['Thời gian'].apply(parse_hoadon_date_strict)
df_hoadon['Doanh Thu (VNĐ)'] = df_hoadon['Khách cần trả'].apply(clean_money)

# Lọc cột Hẹn & Hóa đơn
df_hen = df_hen[['Phone_Clean', 'NGÀY HẸN']].dropna(subset=['Phone_Clean'])
df_hoadon = df_hoadon[['Phone_Clean', 'Mã hóa đơn', 'Mã khách hàng', 'Tên hàng', 'Thời_gian_Chuẩn', 'Doanh Thu (VNĐ)']].dropna(subset=['Phone_Clean'])

# ==========================================
# 4. CHỐNG TRÙNG LẶP TRƯỚC MERGE
# ==========================================
# Lấy dòng Lead sớm nhất của mỗi khách hàng
df_lead_unique = df_lead.sort_values('Ngay_Lead_Chuẩn', na_position='last').drop_duplicates('Phone_Clean', keep='first').copy()
df_lead_unique['In_Lead'] = True  

df_hen_unique = df_hen.drop_duplicates(subset=['Phone_Clean'], keep='first').copy()
df_hen_unique['In_Hen'] = True    

# Tách nhóm Lead vãng lai không có SĐT
df_lead_no_phone = df_lead[df_lead['Phone_Clean'].isna()].copy()
df_lead_no_phone['In_Lead'] = True
df_lead_no_phone['Phân nhóm MECE'] = 'Nhóm 0: Lead chưa có SĐT'
df_lead_no_phone['Số lượng Lead tính'] = 1 

# ==========================================
# 5. MERGE OUTER GỘP 3 FILE
# ==========================================
print("Đang gộp dữ liệu và áp dụng bộ lọc thời gian thông minh...")
df_merge = pd.merge(df_lead_unique.dropna(subset=['Phone_Clean']), df_hoadon, on='Phone_Clean', how='outer')
df_final = pd.merge(df_merge, df_hen_unique, on='Phone_Clean', how='outer')

# ==========================================
# 6. BỘ LỌC THỜI GIAN THÔNG MINH (CHẶN 2025, GIỮ LẠI LEAD RỚT 2026)
# ==========================================
cond_hoadon_2026 = df_final['Thời_gian_Chuẩn'] >= '2026-01-01'
cond_lead_rot = df_final['Thời_gian_Chuẩn'].isna() 
df_final = df_final[cond_hoadon_2026 | cond_lead_rot].copy()

# ==========================================
# 7. PHÂN NHÓM MECE & LÀM SẠCH THUỘC TÍNH (NGUỒN, SP)
# ==========================================
def phan_nhom_mece(row):
    L = row['In_Lead'] == True
    H = row['In_Hen'] == True
    HD = pd.notna(row['Mã hóa đơn'])
    if not L and not H and HD: return 'Nhóm 1: Khách tự nhiên vãng lai (Chỉ có HĐ)'
    elif L and not H and not HD: return 'Nhóm 2: Lead có SĐT nhưng chưa hẹn & chưa chốt'
    elif L and not H and HD: return 'Nhóm 3: Lead chốt thẳng không cần hẹn'
    elif L and H and not HD: return 'Nhóm 4: Lead đặt hẹn nhưng rớt (bom lịch/ko mua)'
    elif L and H and HD: return 'Nhóm 5: Lead hoàn hảo (Đủ 3 bước)'
    else: return 'Khác'

df_final['Phân nhóm MECE'] = df_final.apply(phan_nhom_mece, axis=1)

# Xử lý Nguồn, Sản Phẩm (Giữ nguyên logic cực tốt của bạn từ script cũ)
idx_nhom1 = df_final['Phân nhóm MECE'].str.contains('Nhóm 1', na=False)
df_final.loc[idx_nhom1 & df_final['LOẠI TIN NHẮN'].isna(), 'LOẠI TIN NHẮN'] = 'Khác'
df_final.loc[idx_nhom1 & df_final['NGUỒN'].isna(), 'NGUỒN'] = 'Khác'

def fill_nhom_sp(row):
    if pd.notna(row['NHÓM SP']): return str(row['NHÓM SP']).strip().upper()
    th = str(row['Tên hàng']).upper()
    if any(x in th for x in ['KHÓA HỌC', 'ĐÀO TẠO', 'PLASMA ONLINE']): return 'ĐÀO TẠO'
    return 'DỊCH VỤ'
df_final['NHÓM SP_Clean'] = df_final.apply(fill_nhom_sp, axis=1)

def phan_loai_nguon(row):
    cmb = (str(row['CHATPAGE']) + " | " + str(row['NGUỒN'])).upper()
    if any(x in cmb for x in ['FB CÔ HƯỜNG', 'FANPAGE', 'FACEBOOK', 'FB']): return 'Facebook'
    if any(x in cmb for x in ['TIKTOK']): return 'Tiktok'
    if any(x in cmb for x in ['HOTLINE', 'ZALO']): return 'Hotline/Zalo'
    if any(x in cmb for x in ['KHÁCH CŨ', 'DATA CŨ 2023', 'FILE CŨ']): return 'Khách cũ'
    if any(x in cmb for x in ['VÃNG LAI']) or ('NHÓM 1' in row['Phân nhóm MECE'].upper()): return 'Vãng lai'
    return 'Khác'
df_final['Kênh Tiếp Cận'] = df_final.apply(phan_loai_nguon, axis=1)

def lay_sp_cot_loi(row):
    sp = (str(row['Tên hàng']) if pd.notna(row['Tên hàng']) else str(row['QUAN TÂM'])).upper()
    if 'XOÁ' in sp or 'XÓA' in sp: return 'Xóa Laser'
    if any(x in sp for x in ['MÀY', 'TẠO SỢI', 'ĐIÊU KHẮC']): return 'Làm Mày'
    if 'MÔI' in sp: return 'Làm Môi'
    if 'CO2' in sp or 'BÓC TÁCH' in sp: return 'CO2 / Cắt đáy sẹo'
    if 'PLASMA' in sp and 'KHÓA' in sp: return 'Khóa học'
    if 'MÍ' in sp or 'EYELINER' in sp: return 'Làm Mí'
    if sp == 'NAN': return 'Chưa rõ'
    return 'Dịch vụ/Sản phẩm khác'
df_final['Phân loại sản phẩm'] = df_final.apply(lay_sp_cot_loi, axis=1)

# ==========================================
# 8. CHỐNG NHÂN BẢN ẢO PHỄU (CHỈ ĐẾM LEAD 1 LẦN)
# ==========================================
print("Đang xử lý chống nhân bản ảo số liệu Funnel...")
df_final = df_final.sort_values(by=['Phone_Clean', 'Thời_gian_Chuẩn'], na_position='first')
is_first_interaction = ~df_final.duplicated(subset=['Phone_Clean'], keep='first')

# Định nghĩa các bước phễu
def get_funnel_b2(row): return 0 if 'Nhóm 0' in row['Phân nhóm MECE'] else 1
def get_funnel_b3(row): return 1 if any(x in row['Phân nhóm MECE'] for x in ['Nhóm 4', 'Nhóm 5']) or pd.notna(row['NGÀY HẸN']) else 0
def get_funnel_b4(row): return 1 if any(x in row['Phân nhóm MECE'] for x in ['Nhóm 3', 'Nhóm 5', 'Nhóm 1']) or pd.notna(row['Mã hóa đơn']) else 0

df_final['[F] 1_Có Inbox'] = 1
df_final['[F] 2_Có SĐT'] = df_final.apply(get_funnel_b2, axis=1)
df_final['[F] 3_Có Đặt Lịch'] = df_final.apply(get_funnel_b3, axis=1)
df_final['[F] 4_Có Ra Đơn'] = df_final.apply(get_funnel_b4, axis=1)
df_final['Số lượng Lead tính'] = 1

# Các dòng hóa đơn mua thêm (sau hóa đơn đầu tiên) -> ép phễu về 0
for col in ['[F] 1_Có Inbox', '[F] 2_Có SĐT', '[F] 3_Có Đặt Lịch', '[F] 4_Có Ra Đơn', 'Số lượng Lead tính']:
    df_final.loc[~is_first_interaction, col] = 0

# ==========================================
# 9. TÍNH TỐC ĐỘ CHỐT ĐƠN VÀ ĐỒNG BỘ HIỂN THỊ
# ==========================================
df_final['Thời gian ra đơn (Ngày)'] = (df_final['Thời_gian_Chuẩn'] - df_final['Ngay_Lead_Chuẩn']).dt.total_seconds() / (24 * 3600)
df_final['Thời gian ra đơn (Ngày)'] = df_final['Thời gian ra đơn (Ngày)'].round(1)
df_final.loc[df_final['Thời gian ra đơn (Ngày)'] < 0, 'Thời gian ra đơn (Ngày)'] = 0

# Đồng bộ định dạng Datetime thành chuỗi dễ đọc YYYY-MM-DD HH:MM:SS
df_final['Thời_gian_Chuẩn'] = pd.to_datetime(df_final['Thời_gian_Chuẩn'])
df_final['Ngay_Lead_Chuẩn'] = pd.to_datetime(df_final['Ngay_Lead_Chuẩn'])

df_final['SĐT Cuối'] = df_final['Phone_Clean'].apply(lambda x: f"'{x}" if pd.notna(x) else np.nan)

# ==========================================
# 10. GỘP LEAD KHÔNG SĐT VÀ XUẤT FILE
# ==========================================
# Dọn dẹp Nhóm 0 (Lead ko SĐT) trước khi nối vào
df_lead_no_phone['[F] 1_Có Inbox'] = 1
for col in ['[F] 2_Có SĐT', '[F] 3_Có Đặt Lịch', '[F] 4_Có Ra Đơn', 'Doanh Thu (VNĐ)']: df_lead_no_phone[col] = 0
df_lead_no_phone['Ngay_Lead_Chuẩn'] = pd.to_datetime(df_lead_no_phone['NGÀY'].apply(parse_lead_date_strict))

df_export = pd.concat([df_final, df_lead_no_phone], ignore_index=True)

# Xóa các cột kỹ thuật
cols_to_drop = ['In_Lead', 'In_Hen', 'Phone_Clean', 'SỐ ĐT', 'NGÀY']
df_export.drop(columns=[c for c in cols_to_drop if c in df_export.columns], inplace=True)

# Đưa các cột quan trọng lên đầu
first_cols = ['SĐT Cuối', 'Ngay_Lead_Chuẩn', 'Thời_gian_Chuẩn', 'Phân nhóm MECE', 'Số lượng Lead tính', 'Thời gian ra đơn (Ngày)']
df_export = df_export[first_cols + [c for c in df_export.columns if c not in first_cols]]

# Lưu file
out_path = '/Users/tranvomanhtuan/Downloads/Data_Dashboard_FINAL_PIPELINE.xlsx'
with pd.ExcelWriter(out_path, datetime_format='DD/MM/YYYY') as writer:
    df_export.to_excel(writer, index=False)

print(f"\n🎉 HOÀN TẤT! File Sạch 100% đã sẵn sàng để up Looker Studio.")
print(f"Tổng số dòng: {len(df_export)} | Doanh thu ảo 2025 đã bị xóa | Không còn đếm trùng Lead.")
print(f"Lưu tại: {out_path}")

Đang đọc và làm sạch dữ liệu thô...


/var/folders/m5/m62mtz3d5172h5lxcfxt6ff40000gn/T/ipykernel_44114/3468616490.py:38: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  d = pd.to_datetime(hd_time, errors='coerce')


Đang gộp dữ liệu và áp dụng bộ lọc thời gian thông minh...
Đang xử lý chống nhân bản ảo số liệu Funnel...

🎉 HOÀN TẤT! File Sạch 100% đã sẵn sàng để up Looker Studio.
Tổng số dòng: 5021 | Doanh thu ảo 2025 đã bị xóa | Không còn đếm trùng Lead.
Lưu tại: /Users/tranvomanhtuan/Downloads/Data_Dashboard_FINAL_PIPELINE.xlsx


In [2]:
import pandas as pd

print("==========================================================")
print("🔍 BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY CHECK)")
print("==========================================================\n")

# Đọc file kết quả sau khi chạy Pipeline (Hãy đảm bảo đường dẫn này đúng)
file_path = '/Users/tranvomanhtuan/Downloads/Data_Dashboard_FINAL_PIPELINE.xlsx'
try:
    df_check = pd.read_excel(file_path)
    print(f"Đã nạp file thành công. Tổng số dòng dữ liệu: {len(df_check)}\n")
except Exception as e:
    print(f"❌ Lỗi đọc file: {e}")
    exit()

# ---------------------------------------------------------
# TEST 1: KIỂM TRA LỖI LỌT DOANH THU 2025
# ---------------------------------------------------------
df_check['Thời_gian_Chuẩn_dt'] = pd.to_datetime(df_check['Thời_gian_Chuẩn'], errors='coerce')
years_found = df_check['Thời_gian_Chuẩn_dt'].dt.year.dropna().unique()

print("TEST 1: Kiểm tra ranh giới thời gian (Time Boundary)")
print(f"- Các năm có phát sinh doanh thu trong file: {list(years_found)}")
if 2025 in years_found:
    print("  ❌ LỖI NGHIÊM TRỌNG: Vẫn còn lọt hóa đơn năm 2025 vào báo cáo!")
else:
    print("  ✅ PASS: Doanh thu ảo của năm 2025 đã bị dọn sạch.")

# ---------------------------------------------------------
# TEST 2: KIỂM TRA BẢO TOÀN KHÁCH RỚT PHỄU
# ---------------------------------------------------------
lead_rot_pheu = df_check['Thời_gian_Chuẩn'].isna().sum()
print("\nTEST 2: Kiểm tra dữ liệu Lead chưa chốt đơn")
print(f"- Số dòng Lead không có ngày hóa đơn: {lead_rot_pheu} dòng")
if lead_rot_pheu == 0:
    print("  ❌ LỖI NGHIÊM TRỌNG: Tất cả khách rớt phễu đã bị xóa nhầm!")
else:
    print("  ✅ PASS: Dữ liệu khách hàng chưa chốt được bảo toàn hoàn hảo.")

# ---------------------------------------------------------
# TEST 3: KIỂM TRA NHÂN BẢN PHỄU (DEDUPLICATION)
# ---------------------------------------------------------
# Bỏ qua nhóm không có SĐT (Vãng lai) vì nhóm này vốn dĩ đã là các dòng độc lập
df_co_sdt = df_check[df_check['SĐT Cuối'].notna()]
max_inbox = df_co_sdt.groupby('SĐT Cuối')['[F] 1_Có Inbox'].sum().max()
max_don = df_co_sdt.groupby('SĐT Cuối')['[F] 4_Có Ra Đơn'].sum().max()

print("\nTEST 3: Kiểm tra đếm trùng lặp Phễu")
print(f"- Số lượt tính 'Inbox' tối đa cho 1 SĐT: {max_inbox}")
print(f"- Số lượt tính 'Ra đơn' tối đa cho 1 SĐT: {max_don}")

if max_inbox > 1 or max_don > 1:
    print("  ❌ LỖI NGHIÊM TRỌNG: Có khách hàng bị đếm nhiều lần vào Phễu!")
else:
    print("  ✅ PASS: Mỗi SĐT chỉ được đếm 1 lần duy nhất, chống nhân bản thành công.")

# ---------------------------------------------------------
# TEST 4: TỔNG KẾT CON SỐ THẬT
# ---------------------------------------------------------
total_rev = df_check['Doanh Thu (VNĐ)'].sum()
total_inbox = df_check['[F] 1_Có Inbox'].sum()

print("\n📊 TỔNG KẾT SỐ LIỆU ĐƯA LÊN LOOKER STUDIO:")
print(f"- Tổng Doanh Thu 2026: {total_rev:,.0f} VNĐ")
print(f"- Tổng Khách Inbox: {total_inbox:,.0f} khách")
print("==========================================================")

🔍 BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY CHECK)

Đã nạp file thành công. Tổng số dòng dữ liệu: 5021

TEST 1: Kiểm tra ranh giới thời gian (Time Boundary)
- Các năm có phát sinh doanh thu trong file: []
  ✅ PASS: Doanh thu ảo của năm 2025 đã bị dọn sạch.

TEST 2: Kiểm tra dữ liệu Lead chưa chốt đơn
- Số dòng Lead không có ngày hóa đơn: 5021 dòng
  ✅ PASS: Dữ liệu khách hàng chưa chốt được bảo toàn hoàn hảo.

TEST 3: Kiểm tra đếm trùng lặp Phễu
- Số lượt tính 'Inbox' tối đa cho 1 SĐT: 1
- Số lượt tính 'Ra đơn' tối đa cho 1 SĐT: 1
  ✅ PASS: Mỗi SĐT chỉ được đếm 1 lần duy nhất, chống nhân bản thành công.

📊 TỔNG KẾT SỐ LIỆU ĐƯA LÊN LOOKER STUDIO:
- Tổng Doanh Thu 2026: 2,504,224,000 VNĐ
- Tổng Khách Inbox: 4,822 khách
